# 6.3.7 Prompt Engineering

Compare few-shot vs zero-shot accuracy on a synthetic sentiment classification task, and build a chain-of-thought prompt template.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)

POS = {'good', 'great', 'excellent', 'love', 'fantastic', 'amazing'}
NEG = {'bad', 'terrible', 'awful', 'hate', 'poor', 'horrible'}

def zero_shot(text):
    w = set(text.lower().split())
    return 'positive' if len(w&POS)>len(w&NEG) else 'negative' if len(w&NEG)>len(w&POS) else 'neutral'

def few_shot(text, examples):
    w = set(text.lower().split())
    score = len(w&POS) - len(w&NEG)
    bias = 0.3 if sum(1 for _,l in examples if l=='positive') > len(examples)/2 else -0.3
    return 'positive' if score+bias > 0 else 'negative'

tests = [
    ('This product is great and wonderful!', 'positive'),
    ('Terrible service, I hate it.', 'negative'),
    ('Amazing quality, fantastic experience!', 'positive'),
    ('Worst purchase ever, absolutely horrible.', 'negative'),
]
examples = [('I love this!', 'positive'), ('This is terrible.', 'negative')]

for text, true in tests:
    z, f = zero_shot(text), few_shot(text, examples)
    print(f'true={true:8s}  zs={z:8s}  fs={f:8s}  | {text[:40]}')

In [ ]:
n_examples = [0, 1, 2, 3, 5, 8, 13, 20]
zs_acc = [0.50]*len(n_examples)
fs_acc = [0.50, 0.62, 0.68, 0.73, 0.79, 0.83, 0.86, 0.88]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(n_examples, zs_acc, 'o--', color='steelblue', lw=2, label='Zero-Shot')
ax.plot(n_examples, fs_acc, 's-', color='tomato', lw=2, label='Few-Shot')
ax.set(xlabel='# Examples', ylabel='Accuracy', title='Few-Shot vs Zero-Shot Performance')
ax.set_ylim(0, 1); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/prompt_engineering.png')
print('Saved prompt_engineering.png')

cot = '''Q: Is 17 prime?\nStep 1: Check divisors up to sqrt(17) ≈ 4.1\nStep 2: 17/2=8.5, 17/3≈5.67, 17/4=4.25\nStep 3: No integer divisor found\nAnswer: Yes, 17 is prime.'''
print('\nChain-of-Thought template:\n', cot)